# Importing auxiliary packages

In [ ]:
!git clone https://github.com/gv-americas/ml_course_americas.git
%cd ml_course_americas

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt

# Redefining the Grid class

In [ ]:
class Grid:
    def __init__(self, ox=None, oy=None, nx=None, ny=None, sx=None, sy=None):
        self.ox = ox
        self.oy = oy

        self.nx = nx
        self.ny = ny

        self.sx = sx
        self.sy = sy

    def from_data(self, x, y, sx, sy):
        self.sx = sx
        self.sy = sy
        self.ox, self.oy = np.min(x), np.min(y)
        maxx, maxy = np.max(x)+self.sx, np.max(y)+self.sy
        self.nx, self.ny = int(np.ceil((maxx-self.ox)/sx)), int(np.ceil((maxy-self.oy)/sy))

    def get_coords(self):
        x_coord = np.arange(self.nx) * self.sx + self.ox - self.sx/2
        y_coord = np.arange(self.ny) * self.sy + self.oy - self.sy/2

        return np.array([[x, y] for y, x in product(y_coord, x_coord)])

# Creating a function to plot grids

In [ ]:
def pixel_plot(grid, variable, point_x, point_y, point_variable, figsize=(10,10)):
    
    tg_coords = grid.get_coords()
    
    fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(variable.reshape(grid.ny, grid.nx), cmap='jet', extent=(np.min(tg_coords.T[0])-grid.sx/2, np.max(tg_coords.T[0])+grid.sx/2, np.min(tg_coords.T[1])-grid.sy/2, np.max(tg_coords.T[1])+grid.sy/2), origin='lower')

    ax.scatter(point_x, point_y, c=point_variable, cmap='jet', edgecolors='black', s=20)

    #ax.scatter(tg_coords.T[0], tg_coords.T[1], color='white', s=1) # to check if the grid is in correct place

    ax.set(
    title = 'Pixel plot',
    xlabel = 'X (m)',
    ylabel='Y (m)',
    aspect = 'equal' #important for grid plots
    )

    ax.minorticks_on()
    ax.grid(linestyle='--', which='major')

    # Create an axs for colorbar. The position of the axs is calculated based on the position of ax.
    # You can change 0.01 to adjust the distance between the main image and the colorbar.
    # You can change 0.02 to adjust the width of the colorbar.
    # This practice is universal for both subplots and Geoaxs.
    # https://stackoverflow.com/questions/18195758/set-matplotlib-colorbar-size-to-match-graph

    cax = fig.add_axes([ax.get_position().x1+0.01,ax.get_position().y0,0.02,ax.get_position().height])
    plt.colorbar(im, cax=cax, label='Grades %') # Similar to fig.colorbar(im, cax = cax)

    plt.show()

# Importing data

In [ ]:
walker = pd.read_csv('00_Data/walker_lake_cat.csv')

# Creating a grid

In [ ]:
grid = Grid()

In [ ]:
grid.from_data(walker.X, walker.Y, 5, 5)

In [ ]:
tg_coords = grid.get_coords()

# Interpolation

https://docs.scipy.org/doc/scipy/reference/interpolate.html

In [ ]:
from scipy import interpolate

## Getting the variables we need for interpolation

In [ ]:
xobs = np.array([walker['X'], walker['Y']]).T
yobs = walker['V']
xflat = tg_coords

## Intepolating using RBF

In [ ]:
yflat = interpolate.RBFInterpolator(xobs, yobs, kernel='linear')(xflat)

## Plotting the results

In [ ]:
pixel_plot(grid, yflat, walker['X'], walker['Y'], walker['V'], figsize=(10,10))

# Spatial

In [ ]:
from scipy import spatial

## Pdist

https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.pdist.html

* Takes a single array of points as input.
* Calculates all pairwise distances between the points (excluding duplicates).
* Returns a condensed distance matrix (shape: (n_samples * (n_samples - 1)) / 2).

In [ ]:
dists = spatial.distance.pdist(xobs)

In [ ]:
dists.shape

## Cdist

https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html

* Can take two input arrays of points.
* Calculates pairwise distances between points in different arrays.
* Returns a full distance matrix (shape: (n_samples1, n_samples2)).

In [ ]:
dists = spatial.distance.cdist(xobs, xobs)

In [ ]:
dists.shape

## KDTree

https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.KDTree.html

https://www.youtube.com/watch?v=BK5x7IUTIyU

In [ ]:
ref_pt = (100, 150)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

scatvu = ax.scatter(walker['X'], walker['Y'], c=walker['V'], cmap='jet')
ax.scatter([ref_pt[0]], [ref_pt[1]], s=80, marker='x', color='red')

ax.set(
title = 'Location map for V',
xlabel = 'X (m)',
ylabel='Y (m)',
aspect = 'equal' #importante para mapas de localizacao
)

ax.minorticks_on()
ax.grid(linestyle='--', which='major')

fig.colorbar(scatvu, label='Grade V')

plt.show()

## Creating the tree

In [ ]:
tree = spatial.KDTree(xobs)

## Query

Query the kd-tree for nearest neighbors.

In [ ]:
nn = 100

In [ ]:
dists, idxs = tree.query(np.array([[ref_pt[0], ref_pt[1]]]), nn)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

scatvu = ax.scatter(walker['X'], walker['Y'], c='gray', alpha=0.3)
ax.scatter(walker['X'][idxs[0]], walker['Y'][idxs[0]], color='red')
ax.scatter([ref_pt[0]], [ref_pt[1]], s=80, marker='x', color='red')

ax.set(
title = 'Location map for V',
xlabel = 'X (m)',
ylabel='Y (m)',
aspect = 'equal' #importante para mapas de localizacao
)

ax.minorticks_on()
ax.grid(linestyle='--', which='major')

plt.show()

## Query ball point

Find all points within distance r of point(s) x.

In [ ]:
d = 100

In [ ]:
idxs = tree.query_ball_point(np.array([[ref_pt[0], ref_pt[1]]]), d)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

scatvu = ax.scatter(walker['X'], walker['Y'], c='gray', alpha=0.3)
circle = plt.Circle(ref_pt, d, color='red', alpha=0.5)
ax.add_patch(circle)
ax.scatter(walker['X'][idxs[0]], walker['Y'][idxs[0]], color='red')
ax.scatter([ref_pt[0]], [ref_pt[1]], s=80, marker='x', color='red')


ax.set(
title = 'Location map for V',
xlabel = 'X (m)',
ylabel='Y (m)',
aspect = 'equal' #importante para mapas de localizacao
)

ax.minorticks_on()
ax.grid(linestyle='--', which='major')

plt.show()

# Kriging with Python

## Defining the covariance function

In [ ]:
def exponetial(h, r, c0, b=0):
    a = r/3
    return c0 - (b + c0 * (1 - np.exp(-h/a)))

### Defining covariance model parameters

In [ ]:
r = 160. #range
ne = 0.0 #nugget effect

## Refreshing your memory about the math

$$
    \begin{bmatrix}
    C_{11}&\dots&C_{1n}&1\\
    \vdots&\ddots&\vdots&1\\
    C_{n1}&\dots&C_{nn}&1\\
    1&1&1&0
    \end{bmatrix}
    \begin{bmatrix}
    \lambda_{1}\\
    \vdots\\
    \lambda_{n}\\
    \mu
    \end{bmatrix}
    =
    \begin{bmatrix}
    C_{10}\\
    \vdots\\
    C_{n0}\\
    1
    \end{bmatrix}
$$

## Doing the calculations

In [ ]:
kn = 100 #setting number of neighbord per estimation
results = [] #defining an empty list for storing the results

#iterating through each node of the grid
for node in tg_coords:
    
    #querying our existing tree for the closest kn samples
    dists_data_u, idxs = tree.query(node, kn)

    ##

    #creating the left hand side matrix
    Cnn = np.ones((kn + 1, kn + 1))
    Cnn[-1, -1] = 0.0

    #calculating distances between samples
    data_locs = np.array([walker['X'][idxs], walker['Y'][idxs]]).T
    
    dists_data_data = spatial.distance.cdist(data_locs, data_locs)

    #calculating covariances from distances
    covs_data_data = exponetial(dists_data_data, r, 1-ne, ne)

    #filling the ones matrix
    Cnn[:-1, :-1] = covs_data_data

    ###

    #creating the right hand side matrix
    Cn0 = np.ones(kn + 1)

    #calculating covariances from distancess
    covs_data_u = exponetial(dists_data_u, r, 1-ne, ne)

    #filling the ones matrix
    Cn0[:-1] = covs_data_u

    ##

    #calculating the weights
    #weights = (np.linalg.inv(Cnn)@Cn0)[:-1]

    #we can do the same faster
    weights = np.linalg.solve(Cnn, Cn0)[:-1]

    #linear combination between samples and weights
    result = walker['V'][idxs]@weights

    results.append(result)

## Ploting the results

In [ ]:
pixel_plot(grid, np.array(results), walker['X'], walker['Y'], walker['V'], figsize=(10,10))

# Exercise

Create a function that receives as arguments a grid object, coordinates X and Y and a variable V and a radius r. This function returns an array with the results of a IDW interpolation to all grid node centroids. Plot the results and compare with kriging results using a scatterplot. 